In [0]:
# COMMAND ----------

import pandas as pd
import os

# 1. RUTA: Apuntando a la carpeta 'Datos' del repositorio clonado en tu Workspace
folder_path = "/Workspace/Users/u20216288@utp.edu.pe/Project-SQL-Caso_CancelacionesHoteleras-RADL/Data"

# 2. ESQUEMA: Destino en proyecto_kiosko
catalog_schema = "cursodataacademy.project_hotel"

# Buscamos los archivos CSV en la carpeta del repositorio
csv_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".csv")]

print(f"Se encontraron {len(csv_files)} archivos CSV:\n")

for csv_file in sorted(csv_files):
    file_path = os.path.join(folder_path, csv_file)
    table_name = csv_file.replace(".csv", "").lower()
    full_table_name = f"{catalog_schema}.{table_name}"

    # 3. MANEJO DE SEPARADORES: En la tabla cancelaciones hoteleras se usa ','
    separador = ","

    # Leer CSV con el separador dinámico
    df_pandas = pd.read_csv(file_path, encoding="utf-8-sig", sep=separador, skipinitialspace=True)

    # Reemplaza los espacios y caracteres problemáticos por guiones bajos
    df_pandas.columns = df_pandas.columns.str.replace(r'[ ,;{}()\n\t=]', '_', regex=True)

    # Crear DataFrame de Spark y guardar como tabla Delta
    df_spark = spark.createDataFrame(df_pandas)
    df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)

    row_count = df_spark.count()
    col_count = len(df_spark.columns)
    print(f"✅ {csv_file} → {full_table_name} ({row_count} filas, {col_count} columnas: {df_spark.columns})")

print(f"\n✨ Carga completa: {len(csv_files)} tablas creadas en {catalog_schema}")

In [0]:
%sql
--Creacion de la tabla tb_reservas
CREATE OR REPLACE TABLE cursodataacademy.project_hotel.tb_reservas AS
SELECT 
    hotel,
    is_canceled,
    lead_time,
    arrival_date_year,
    arrival_date_month,
    arrival_date_day_of_month,
    distribution_channel,
    reservation_status_date,
    adr,
    children,
    babies,
    total_of_special_requests,
    booking_changes
FROM cursodataacademy.project_hotel.hotel_bookings;

In [0]:
%sql --Creacion de la tabla tb_reservas_prueba
CREATE OR REPLACE TABLE cursodataacademy.project_hotel.tb_reservas_prueba AS
SELECT * 
FROM cursodataacademy.project_hotel.tb_reservas

In [0]:
%sql 
--Crear la columna arrival_Date para ejercicio 9
ALTER TABLE cursodataacademy.project_hotel.tb_reservas
ADD COLUMN arrival_date DATE;


In [0]:
%sql 
--Actualizar la columna arrival_date como fecha
UPDATE cursodataacademy.project_hotel.tb_reservas
SET arrival_date = TO_DATE(
                            CONCAT_WS('-',
                              CAST(arrival_date_day_of_month AS STRING),
                              CAST(arrival_date_month AS STRING),
                              CAST(arrival_date_year AS STRING)),
                            'd-MMMM-yyyy');